#### 문서의 내용을 읽고 쪼개기

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size= 1500,       ## 하나의 청크가 가질 토큰 수(청크 크기)
  chunk_overlap= 200      ## 청크 간에 중복시킬 토큰 수
)

loader= Docx2txtLoader('Tax.docx')


document_list  = loader.load_and_split(text_splitter= splitter)

document_list

#### 문서 임베딩 후 벡터 데이터베이스로 저장

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
%pip install langchain-upstage

In [8]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model = 'embedding-passage')

#### ChromaDB로 임베디드 DB 만들기

In [ ]:
%pip install langchain-chroma

In [9]:
from langchain_chroma import Chroma

# 처음 DB 생성할 때는 이 방식으로

database = Chroma.from_documents(
  documents= document_list,
  embedding=embedding, 
  collection_name='chroma-tax',
  persist_directory= 'upstage_chroma'
)

# database = Chroma(
#   collection_name= 'chroma-tax',
#   persist_directory= './upstage_croma',
#   embedding_function= embedding     # 요거 안쓰면 Chroma 기본 차원수 384
# )

#### Retrieve

In [20]:
query = '기타소득의 세율을 기타소득의 종류별로 설명해주세요'

retrieved_docs = database.similarity_search(query=query, k=20)

In [11]:
retrieved_docs

[Document(id='5b9a3b39-3c40-4bbc-bac4-b7838fb551f6', metadata={'source': 'Tax.docx'}, page_content='다. 제16조제1항제10호에 따른 직장공제회 초과반환금에 대해서는 기본세율\n\n라. 그 밖의 이자소득에 대해서는 100분의 14\n\n2. 배당소득에 대해서는 다음에 규정하는 세율\n\n가. 제17조제1항제8호에 따른 출자공동사업자의 배당소득에 대해서는 100분의 25\n\n나. 그 밖의 배당소득에 대해서는 100분의 14\n\n3. 원천징수대상 사업소득에 대해서는 100분의 3. 다만, 외국인 직업운동가가 한국표준산업분류에 따른 스포츠 클럽 운영업 중 프로스포츠구단과의 계약에 따라 용역을 제공하고 받는 소득에 대해서는 100분의 20으로 한다.\n\n4. 근로소득에 대해서는 기본세율. 다만, 일용근로자의 근로소득에 대해서는 100분의 6으로 한다.\n\n5. 공적연금소득에 대해서는 기본세율\n\n5의2.제20조의3제1항제2호나목 및 다목에 따른 연금계좌 납입액이나 운용실적에 따라 증가된 금액을 연금수령한 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 각 목의 요건을 동시에 충족하는 때에는 낮은 세율을 적용한다.\n\n가. 연금소득자의 나이에 따른 다음의 세율\n\n\n\n나. 삭제<2014. 12. 23.>\n\n다. 사망할 때까지 연금수령하는 대통령령으로 정하는 종신계약에 따라 받는 연금소득에 대해서는 100분의 3\n\n5의3. 제20조의3제1항제2호가목에 따라 퇴직소득을 연금수령하는 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 연금 실제 수령연차 및 연금외수령 원천징수세율의 구체적인 내용은 대통령령으로 정한다.\n\n가. 연금 실제 수령연차가 10년 이하인 경우: 연금외수령 원천징수세율의 100분의 70\n\n나. 연금 실제 수령연차가 10년을 초과하고 20년 이하인 경우: 연금외수령 원천징수세율의 100분의 60\n\n다. 연금 실제 수령연

#### Augmented Generation

In [12]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model = 'claude-3-haiku-20240307')

In [13]:
prompt = f'''[Identity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [Context]를 참고하여 사용자의 [Question]에 답변해주세요.

[Context]
{retrieved_docs}

[Question]
{query}
'''

In [14]:
ai_message= llm.invoke(prompt)

In [15]:
ai_message.content

'기타소득에 대한 세율은 다음과 같습니다:\n\n1. 제14조제3항제8호라목 및 마목에 해당하는 소득금액이 3억원을 초과하는 경우 그 초과하는 분에 대해서는 100분의 30\n2. 제21조제1항제18호 및 제21호에 따른 기타소득에 대해서는 100분의 15\n3. 그 밖의 기타소득에 대해서는 100분의 20\n\n다만, 대통령령으로 정하는 봉사료에 대해서는 100분의 5의 세율이 적용됩니다.\n\n또한 제1항을 적용받는 기타소득 중 일부 항목(법인세법 제67조에 따라 기타소득으로 처분된 소득, 연금외수령한 소득 등)에 대해서는 별도의 세율이 정해져 있습니다.\n\n기타소득의 종류와 성격에 따라 다양한 세율이 적용되므로, 구체적인 상황에 따라 정확한 세율을 확인해야 합니다.'

#### LCEL

In [ ]:
%pip install -U langchain langchain-community langchain-core

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# 프롬프트 정의
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 context를 바탕으로 질문에 답하세요.\n\nContext: {context}"),
    ("human", "{question}")
])

# 문서 포맷팅 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 체인 구성
qa_chain = (
    {
        "context": database.as_retriever(k=20) | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 실행
answer = qa_chain.invoke(query)
print(answer)

기타소득의 세율은 다음과 같습니다:

1. 상금, 현상금, 포상금, 보로금 등 - 20% 세율
2. 복권, 경품권 당첨금 - 20% 세율 
3. 사행행위 참가 이익 - 20% 세율
4. 라디오·TV방송 등 해설·계몽·연기 심사 등 용역 소득 - 20% 세율
5. 전문가 용역 제공 소득 - 20% 세율  
6. 그 밖의 고용관계 없는 수당 등 용역 소득 - 20% 세율
7. 「법인세법」 제67조에 따른 기타소득 - 20% 세율
8. 연금외수령 소득 - 15% 세율
9. 주식매수선택권 행사 이익 - 20% 세율
10. 퇴직 후 직무발명보상금 - 20% 세율
11. 뇌물, 알선수재, 배임수재 금품 - 20% 세율
12. 종교관련종사자 종교단체 수령 종교인소득 - 기본세율(근로소득과 동일)

다만, 제14조제3항제8호라목 및 마목에 해당하는 소득금액이 3억원을 초과하는 경우 그 초과하는 부분에 대해서는 30% 세율이 적용됩니다.


In [22]:
query = '사업소득이 있는 자가 받을 수 있는 세액공제에 대해서 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

네, 사업소득이 있는 자가 받을 수 있는 세액공제에 대해 알아보겠습니다.

1. 표준세액공제 (제59조의4 제9항 제2호)
   - 종합소득이 있는 거주자(근로소득이 있는 자는 제외)로서 세액공제 신청을 하지 않은 경우
   - 성실사업자(사업용계좌 신고 등 요건 충족)는 연 12만원 공제
   - 그 외의 경우 연 7만원 공제

2. 기부금 세액공제 (제59조의4 제4항)
   - 이월기부금과 당해연도 기부금을 합산하여 세액공제 가능
   - 근로소득금액과 사업소득금액의 합계액이 3천만원을 초과하는 경우, 그 초과분에 대해 10% 추가공제 (2024년 한시 적용)

3. 연금보험료 공제 (제51조의3)
   - 국민연금보험료, 퇴직연금 등 연금보험료 납부액의 12% 공제

4. 전자계산서 발급·전송에 대한 세액공제 (제56조의3)
   - 전자계산서 발급 및 전송에 따른 비용의 1% 세액공제

5. 외국납부세액공제 (제57조)
   - 국외원천소득에 대해 외국에서 납부한 세액의 일부를 세액공제

이와 같이 사업소득이 있는 자의 경우 다양한 세액공제 제도를 활용할 수 있습니다. 구체적인 적용 요건과 공제 한도 등은 관련 세법 규정을 참고하시기 바랍니다.


In [23]:
query = '2. 전자계산서 발급 세액공제(제56조의3)의 요건에 대해서 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

국세기본법 제163조에 따르면 전자계산서 발급에 대한 규정이 있습니다.

주요 내용은 다음과 같습니다:

1. 전자계산서 발급 의무 대상 사업자
- 「부가가치세법」 제32조제2항에 따른 전자세금계산서 발급 의무 사업자
- 총수입금액 등을 고려하여 대통령령으로 정하는 사업자

2. 전자계산서 발급 사업자는 대통령령으로 정하는 기한까지 전자계산서 발급명세를 국세청장에게 전송해야 합니다.

3. 전자계산서를 발급하지 않아도 되는 경우는 다음과 같습니다:
- 수입재화에 대해 세관장이 발급한 계산서
- 부동산 매각 등 계산서 발급이 적합하지 않은 경우

따라서 전자계산서 발급과 관련된 세액공제 요건은 이와 같은 전자계산서 발급 규정에 부합해야 할 것입니다. 구체적인 요건은 국세기본법 시행령 등 하위 법령에서 정하고 있을 것으로 보입니다.


In [24]:
query = '연봉이 5000만원인 직장인의 산출세액을 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

제공된 정보를 바탕으로 연봉 5,000만원인 직장인의 산출세액은 다음과 같이 계산할 수 있습니다.

1. 근로소득세액공제 계산
- 총급여액이 7,000만원 초과 1억2,000만원 이하인 경우
- 근로소득세액공제 = 66만원 - [(총급여액 - 7,000만원) × 1/2]
            = 66만원 - [(5,000만원 - 7,000만원) × 1/2]
            = 66만원 - (-10만원)
            = 76만원

2. 자녀세액공제 계산
- 2명의 공제대상자녀가 있는 경우
- 자녀세액공제 = 연 55만원

3. 산출세액 계산
- 총 종합소득금액 = 5,000만원
- 산출세액 = 종합소득세액 - 근로소득세액공제 - 자녀세액공제
            = 1,013만원 - 76만원 - 55만원 
            = 882만원

따라서, 연봉 5,000만원인 직장인의 산출세액은 약 882만원입니다.
